In [0]:
source_path = "/Volumes/weather_stream/infra/landing_zone"
checkpoint_path = "/Volumes/weather_stream/infra/streaming_checkpoints"

In [0]:
from pyspark.sql.functions import current_timestamp, col

stream_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .option("cloudFiles.inferColumnTypes", "true")
  .load(source_path)
  .withColumn("ingestion_time", current_timestamp())
)

In [0]:
(stream_df.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .table("weather_stream.bronze.weather_data")
)